In [1]:
file_path = "story1_p1.txt"
text = open(file_path, "r").read()


In [2]:
import re
import spacy
import pandas as pd
from wordfreq import zipf_frequency

nlp = spacy.load("it_core_news_sm")

# Split sentences reliably on . ! ? (the parser alone sometimes merges them)
nlp.add_pipe("sentencizer", before="parser")

# Fixes to spaCy's Italian stopword list:
# it includes "città" (it shouldn't) and misses some articles
nlp.vocab["città"].is_stop = False
for w in ["i", "gli", "le", "lo", "l"]:
    nlp.Defaults.stop_words.add(w)
    nlp.vocab[w].is_stop = True

# Line breaks inside a sentence should not act as separators
clean_text = re.sub(r"\s+", " ", text).strip()
doc = nlp(clean_text)

c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
rows = []
for sentence_id, sentence in enumerate(doc.sents, start=1):
    for tok in sentence:
        if tok.is_alpha:
            rows.append({
                "sentence": sentence_id,
                "word":     tok.lower_,
                "lemma":    tok.lemma_.lower(),
                "pos":      tok.pos_,        # part of speech: NOUN, VERB, ...
                "stopword": tok.is_stop,
                "length":   len(tok.text),   # number of letters
            })

words = pd.DataFrame(rows)
words

,sentence,word,lemma,pos,stopword,length
0,1,sara,sara,PROPN,True,4
1,1,lavorava,lavorare,VERB,False,8
2,1,come,come,ADP,True,4
3,1,botanica,botanica,NOUN,False,8
4,1,università,università,NOUN,False,10
...,...,...,...,...,...,...
746,36,seduta,sedere,VERB,False,6
747,36,ad,a,ADP,True,2
748,36,aspettare,aspettare,VERB,False,9
749,36,alla,a il,ADP,True,4


In [4]:
sentences = pd.DataFrame([
    {
        "sentence": i,
        "n_words":  sum(1 for t in s if t.is_alpha),
        "text":     s.text.strip(),
    }
    for i, s in enumerate(doc.sents, start=1)
])
sentences = sentences[sentences["n_words"] > 0].reset_index(drop=True)
sentences.head()

,sentence,n_words,text
0,1,19,Sara lavorava come botanica all'università di ...
1,2,21,Quell'anno doveva raccogliere alcuni campioni ...
2,3,21,"L'isola si chiamava Malvara, era lunga pochi c..."
3,4,23,Il traghetto partiva ogni mattina alle sette d...
4,5,18,Sara aveva preparato il viaggio con attenzione...


In [5]:
vocab = words.groupby("word", as_index=False).agg(
    count=("word", "size"),
    stopword=("stopword", "first"),
)
vocab["zipf_general"] = [zipf_frequency(w, "it") for w in vocab["word"]]
vocab.sort_values("zipf_general").head()

,word,count,stopword,zipf_general
159,malvara,4,False,0.00
134,gubbana,2,False,0.00
110,fanura,3,False,0.00
154,lusana,2,False,0.00
122,fioriva,1,False,1.84


In [ ]:
vocab.sort_values("zipf_general")